# Fraud Pattern Analysis for a Crypto Exchange
### An Explanatory Data Analysis Consulting Report

**Prepared for:** a mid-sized crypto exchange's Fraud & Compliance team
**Prepared by:** Iman Ashoori - GH1049062, Data Science Consultant ;)

## 1. Business Context

Our client is a mid-sized cryptocurrency exchange that lets users transfer, swap, mint, burn, and bridge tokens across several blockchains (Ethereum, BSC, Polygon, Solana) and platforms (Binance, Coinbase, Kraken, OpenSea, and third-party DEXs/wallets). As trading volume has grown, so has exposure to scams: phishing, rug-pulls, fake token swaps, and cross-chain laundering.

The client's Fraud & Compliance team has a log of 20,000 transactions, each labelled as legitimate or a known scam, along with wallet-level behavioural signals (wallet age, transaction history, velocity, an internal anomaly score). They do not yet have a systematic understanding of *where and how* scam activity concentrates in this data — they have asked us, as data science consultants, to run an exploratory analysis that surfaces concrete, actionable patterns before they invest in a full detection model.

**Dataset source:** [Crypto Scam Transaction Dataset, Kaggle](https://www.kaggle.com/datasets/muhammadhussnain09/crypto-scam-transaction-dataset); a synthetic dataset (20,000 rows x 18 columns) simulating scam and legitimate transaction behaviour, with labels generated via behavioural heuristics rather than confirmed investigations.


In [ ]:
from statistics import correlation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

url = "https://raw.githubusercontent.com/imanashoorii/crypto-fraud-pattern-analysis/main/data/crypto_scam_transaction_dataset.csv"
df = pd.read_csv(url)
df.shape

## 2. Data Exploration

We start by profiling the dataset's structure, data quality, and metadata before doing any analysis.

In [ ]:
df.info()

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
missing_cols = ['gas_fee_usd', 'platform', 'avg_txn_interval_sender_min']
overlap = df[missing_cols].isna().sum(axis=1)
overlap.value_counts().sort_index()

In [ ]:
df['transaction_id'].duplicated().sum()

In [ ]:
for col in ['blockchain', 'transaction_type', 'token_type', 'platform']:
    print(col, '->', df[col].nunique(), 'unique values:', df[col].unique().tolist())

In [ ]:
df['is_scam'].value_counts(normalize=True).round(4) * 100

**class balance check:** the dataset kaggle documentation states a scam rate of 18-22%. The actual computed rate in this dataset is **7.25%**

## 3. Data Preprocessing

Before analysis, we clean and prepaer the dataset:

1. **Timestamps** are stored as raw Unix seconds. We convert them to datetimes and extract `hour` and `day_of_week` which several business questions need them.
2. **Missing values** `platform`, `gas_fee_usd`, and `avg_txn_internal_sender_main` are missing ~3%.
3. **"Unknown DEX" / "Unknown Wallet"** are kept as their own categories
4. **Wallet age bucketing** `sender_wallet_age_days` ranges 0-5 497 days

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()


# Testing the date ranges
print('Date range:', df['day_of_week'].min(), ' to ', df['day_of_week'].max())

In [ ]:
for col in ['gas_fee_usd', 'avg_txn_interval_sender_min']:
    df[col + '_was_missing'] = df[col].isna()
    df[col] = df[col].fillna(df[col].median())

df['platform'] = df['platform'].fillna('Unreported')

df[['gas_fee_usd', 'avg_txn_interval_sender_min', 'platform']].isna().sum()

In [ ]:
bins = [-1, 30, 180, 365, 730, df['sender_wallet_age_days'].max()]
labels = ['New (<30d)', 'Young (30-180d)', 'Established (180-365d)', 'Mature (1-2y)', 'Veteran (2y+)']
df['sender_age_group'] = pd.cut(df['sender_wallet_age_days'], bins, labels=labels)

df['sender_age_group'].value_counts().sort_index()

## 4. Exploratory Data Analysis

### 4.1 Which blockchains and platforms carry the highest scam rates?

**Why this matters:** if scam activity focus on specific chains, teams could prioritize monitoring resources.

### 4.2 Are there time-of-day or day-of-week patterns in scam activity?

**Why this matters:** scam campaings are often automated and may happen in low-oversight hours or specific days.

### 4.3 Do transactions on unidentified ("Unknown") platforms carry more risk?

**Why this matters:** "Unknown DEX" and "Unknown Wallet" represents the exchange cannot positively identify.

### 4.4 Does sender wallet age predict scam risk?

**Why this matters:** wallet age is a signal the client already has at the time of transaction.

### 4.5 Is the client "anomaly_score" actually a relliable indicator of scam risk?

**Why this matter:** the client data already includes "anomaly_score"

In [ ]:
# 4.1

blockchain_rate = pd.crosstab(df['blockchain'], df['is_scam'], normalize='index')[1].sort_values(ascending=False) * 100

platform_rate = pd.crosstab(df['platform'], df['is_scam'], normalize='index')[1].sort_values(ascending=False) * 100

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
blockchain_rate.plot(kind='bar', ax=ax[0], color='blue')
ax[0].set_title('SCAM RATE BY BLOCKCHAIN (%)')
ax[0].set_ylabel('% SCAM RATE')

platform_rate.plot(kind='bar', ax=ax[1], color='red')
ax[1].set_title('SCAM RATE BY PLATFORM (%)')

plt.tight_layout()
plt.show()

print(blockchain_rate.round(2))
print()
print(platform_rate.round(2))


In [ ]:
# 4.2

days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

hourly_rate = pd.crosstab(df['hour'], df['is_scam'], normalize='index')[1] * 100

dow_rate = pd.crosstab(df['day_of_week'], df['is_scam'], normalize='index')[1] * 100
dow_order = days_of_week
dow_rate = dow_rate.reindex(dow_order) * 100

fix, ax = plt.subplots(1, 2, figsize=(12, 4))
hourly_rate.plot(kind='line', marker='o', ax=ax[0], color='orange')
ax[0].set_title("SCAM RATE BY HOURLY EACH DAY (%)")
ax[0].set_xlabel("HOUR (UTC)")

dow_rate.plot(kind='bar', ax=ax[1], color='green')
ax[1].set_title("SCAM RATE BY DAY OF WEEK (%)")

plt.tight_layout()
plt.show()

print('HOUR RANGE:', round(hourly_rate.min(), 2), '-', round(hourly_rate.max(), 2), '%')
print('DAY RANGE:', round(dow_rate.min(), 2), '-', round(dow_rate.max(), 2), '%')


In [ ]:
# 4.3

df['is_unknown_platform'] = df['platform'].isin(['Unknown DEX', 'Unknown Wallet'])

comparison = df.groupby('is_unknown_platform').agg(
    scam_rate_pct=('is_scam', lambda x: x.mean() * 100),
    avg_anomaly_score=('anomaly_score', 'mean'),
    avg_failed_txn_ratio=('failed_txn_ratio_sender', 'mean'),
    avg_velocity_score=('velocity_score', 'mean'),
    avg_amount_usd=('transaction_amount_usd', 'mean'),
    n_transactions=('transaction_id', 'count'),
).round(3)
comparison.index = ['KNOWN PLATFORM', 'UNKNOWN PLATFORM']
comparison

In [ ]:
# 4.4

age_scam_rate = pd.crosstab(df['sender_age_group'], df['is_scam'], normalize='index')[1] * 100

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['crimson' if v > 20 else 'blue' for v in age_scam_rate]
age_scam_rate.plot(kind='bar', ax=ax, color=colors)
ax.set_ylabel('% SCAM RATE')
ax.set_title("SCAM RATE BY WALLET AGE GROUP (%)")
plt.xticks(rotation=30, ha='right')
plt.show()

age_scam_rate.round(2)

In [ ]:
# 4.5

correlations = df[['anomaly_score', 'velocity_score', 'failed_txn_ratio_sender', 'is_scam']].corr()['is_scam'].drop('is_scam')

df['anomaly_quintile'] = pd.qcut(df['anomaly_score'].rank(method='first'), 5, labels=['Q1 (LOWEST)', 'Q2', 'Q3', 'Q4', 'Q5(HIGHEST)'])
quintile_scam_rate = pd.crosstab(df['anomaly_quintile'], df['is_scam'], normalize='index')[1] * 100

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
correlations.plot(kind='barh', ax=ax[0], color='blue')
ax[0].set_title('CORRELATION WITH `is_scam`')
quintile_scam_rate.plot(kind='bar', ax=ax[1], color='red')
ax[1].set_title('SCAM RATE BY `anomaly_score` QUINTILE (%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(correlations.round(2))
print()
print(quintile_scam_rate.round(2))



## 5. Conclusion

### Strength and limitations of this analysis

This analysis draws on a clean, structured dataset (no dup, categorical values), which allowed us to move quicly from exploration to business findings
The main limitation is that the dataset is **fully synthetic**, with scam labels from behavioral heuristics rather than confirmed investigations.

### Key insights

- **Location signals are weak.** Blockchain, platform, time-of-day, and day-of-week all show scam rates within a narrow band (roughly 5-9%), suggesting scam activity here is closer to uniformly distributed than concentrated in specific venues or windows.
- **"Unknown" platforms are not inherently riskier.** Despite the intuitive assumption, unidentified counterparties showed no elevation in scam rate or any other risk signal compared to known platforms.
- **Wallet age is the standout signal.** Wallets under 30 days old have a 45.8% scam rate, roughly 9x every other age group, with a sharp cliff rather than a gradual trend.
- **The client's existing anomaly_score is weak.** It correlates with scam status at only 0.079, underperforming the simpler `failed_txn_ratio_sender` feature (0.158).

### Recommendations for the client

1. **Prioritise wallet age as a real-time risk signal.** Apply stricter friction (transaction limits, step-up verification, manual review) to wallets under 30 days old, this is the single most actionable finding in the analysis.
2. **Do not build monitoring rules around platform, blockchain, or timing**, the data does not support concentrating resources there.
3. **Re-evaluate the internal anomaly_score.** If it currently drives any automated decisions, that deserves review, since a simpler feature already outperforms it.
4. **Treat the dataset's own documentation with caution** going forward, and verify metadata claims (like scam rate) directly against the data before relying on them.
5. **Before any production use, validate these patterns against real investigated cases**, since these labels are heuristic-derived, confirming they hold on real fraud data is a necessary next step before acting on them at scale.
